# Tools 原理

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [2]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習 (Machine Learning, ML) 是一個數據科學領域，它讓電腦系統能夠**從數據中學習，而無需被明確地編程來執行特定任務**。

以下是機器學習的幾個關鍵定義和核心概念：

1.  **Arthur Samuel 的經典定義 (1959 年):**
    > 「機器學習是一個研究領域，它賦予電腦在沒有被明確編程的情況下學習的能力。」
    這是一個廣為人知的早期定義，強調了「無需明確編程」的核心思想。

2.  **Tom M. Mitchell 的現代定義 (1997 年):**
    > 「如果一個電腦程式在**任務 T** 上的性能，由**性能指標 P** 來衡量，隨著**經驗 E** 的增加而提高，那麼我們就說這個程式從經驗中學習了。」
    這個定義更為精確和可操作，它將學習的過程分解為三個關鍵要素：
    *   **經驗 E (Experience):** 指的是程式所接觸到的訓練數據。例如，對於一個圖像識別系統，E 就是大量的帶有標籤的圖像數據。
    *   **任務 T (Task):** 指的是程式需要完成的具體目標。例如，識別圖片中的貓狗，預測房價，翻譯文本等。
    *   **性能指標 P (Performance Measure):** 用於量化程式在任務 T 上表現好壞的標準。例如，分類的準確率、預測的均方誤差、F1 分數等。

**核心思想：**

機器學習的核心在於讓電腦透過分析大量數據來發現其中的模式、趨勢和關聯性，並基於這些發現來構建一個「模型」。這個模型隨後可以用於對新的、未見過的數據做出預測、決策或分類。

**與傳統編程的區別：**

*   **傳統編程：** 人類工程師編寫明確的、一步一步的指令來告訴電腦如何解決問題。當問題複雜或數據量龐大時，編寫這些規則變得非常困難甚至不可能。
    *   *輸入數據 + 規則 (程式碼) = 輸出答案*
*   **機器學習：** 工程師不直接編寫解決問題的規則，而是提供數據和一個學習演算法。演算法會從數據中自動學習規則或模式，然後生成一個模型來解決問題。
    *   *輸入數據 + 答案 (標籤) = 輸出規則 (模型)*
    *   *新輸入數據 + 輸出規則 (模型) = 輸出預測/答案*

**機器學習的主要類型：**

1

# 開始範例

# 串流式對話機器人

In [7]:
system_prompt = '''
你是一個專業的「書店購物AI助手」。你的任務是協助使用者將指定的書籍加入購物車，並確保所有必要資訊（書名與數量）完整無誤。請嚴格遵守以下規則：  
1. 回傳格式
   - 當使用者提供完整資訊（書名與購買數量）時，請直接回傳以下格式：
     ```
     add_to_cart("書名", 數量)
     ```
   - 不得加入任何多餘文字、解釋、評論或格式變化。
   - 書名需以原樣保留（不翻譯、不修改）。

2. 資訊不完整時
   - 若缺少書名或數量，請主動、禮貌地詢問缺失部分。
     - 例：「請問您要購買的書名是什麼？」
     - 例：「請問想購買幾本呢？」

3. 加入購物車前確認
   - 在執行 `add_to_cart()` 前，請再次向使用者確認購買內容。
     - 例：「您確認要將《書名》共 X 本加入購物車嗎？」
   - 僅在使用者明確確認後，才回傳 `add_to_cart("書名", 數量)`。

4. 多本書處理
   - 若使用者同時提供多本書的購買資訊，請逐項確認，並依序回傳多個：
     ```
     add_to_cart("書名1", 數量1)
     add_to_cart("書名2", 數量2)
     ```

5. 禁止輸出
   - 除上述規範內容外，不得輸出任何其他文字、說明或標點。
'''

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
class ChatBot:
    def __init__(self, llm):
        self.llm = llm  # LLM 是對話機器人的大腦
        
        # 初始化對話記錄，包含系統提示詞
        self.messages = [SystemMessage(system_prompt)]

    # 一般版：一次性呼叫模型，整段生成完才回傳
    def chat(self, text):
        # 將使用者輸入加入記錄
        self.messages.append(HumanMessage(text))        
        # 呼叫模型生成完整回覆
        response = self.llm.invoke(self.messages)
        # 將模型回覆加入記錄
        self.messages.append(response)
        # 回傳文字內容
        return response.content

    # 串流版：使用 yield 逐步回傳生成結果
    def chat_stream(self, text):
        # 將使用者輸入加入記錄
        self.messages.append(HumanMessage(text))
        
        # 用於累積完整回覆
        full_response = ""

        # 使用 llm.stream() 逐步獲取生成內容
        for chunk in self.llm.stream(self.messages):
            # chunk.content 為每次生成的一小段文字
            if chunk.content:
                full_response += chunk.content
                # 使用 yield 將部分內容回傳給外部呼叫者
                yield chunk.content

        # 串流結束後，將完整回覆加入記錄
        self.messages.append(AIMessage(full_response))

In [9]:
# 載入 Gradio 套件，用於建立網頁互動介面
import gradio as gr  

# ChatBot 是先前定義的聊天機器人類別
# 以 llm (大型語言模型) 為參數建立一個機器人實例
bot = ChatBot(llm)  

# 定義主要的對話處理函式，支援串流輸出
def chat_function_stream(message, history):
    """
    處理每輪使用者輸入的訊息，並以串流方式回傳回應。
    
    參數：
        message (str)：使用者輸入的文字
        history (list)：對話歷史紀錄，每輪對話包含問答內容
    
    回傳：
        生成器 (generator)：逐步產生回覆的文字片段，可即時在 Gradio 顯示
    """
    # 用於累積完整回覆
    full_response = ""
    
    # 使用 chat_stream 生成器逐步取得模型輸出
    for chunk in bot.chat_stream(message):
        # 將每次生成的片段累加
        full_response += chunk
        # 使用 yield 將目前累積的回覆回傳給 Gradio
        yield full_response

# 建立 Gradio 聊天介面
# - chat_function_stream：每次使用者輸入時呼叫的函式
# - type="messages"：支援多輪對話訊息格式
webui = gr.ChatInterface(chat_function_stream)

# 啟動 Gradio 網頁伺服器
# 使用者可透過瀏覽器訪問本機介面 (例如 http://127.0.0.1:7860)
webui.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
